# NASA

##1.Membaca Dataset

In [ ]:
import pandas as pd

# Path dataset
file_path = "data.csv"

df = pd.read_csv(file_path, sep=';')

# Convert time
df["Timestamp"] = pd.to_datetime(df["time"], unit="s")

# Rename
df.rename(columns={"host": "Host"}, inplace=True)

df.head()

,Unnamed: 0,Host,time,method,url,response,bytes,Timestamp
0,0,***.novo.dk,805465029,GET,/ksc.html,200.0,7067.0,1995-07-11 12:17:09
1,1,***.novo.dk,805465031,GET,/images/ksclogo-medium.gif,200.0,5866.0,1995-07-11 12:17:11
2,2,***.novo.dk,805465051,GET,/images/MOSAIC-logosmall.gif,200.0,363.0,1995-07-11 12:17:31
3,3,***.novo.dk,805465053,GET,/images/USA-logosmall.gif,200.0,234.0,1995-07-11 12:17:33
4,4,***.novo.dk,805465054,GET,/images/NASA-logosmall.gif,200.0,786.0,1995-07-11 12:17:34


##2.Konversi URL Menjadi Label Huruf

In [ ]:
def num_to_letters(n):
    """Konversi angka ke huruf: 0→A, 1→B, ..., 25→Z, 26→AA, dst."""
    result = ""
    n += 1
    while n > 0:
        n -= 1
        result = chr(65 + (n % 26)) + result
        n //= 26
    return result

# --- Hasil Langsung ---
[num_to_letters(i) for i in range(10)]

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']

##3.Pembentukan Web Usage Session

In [ ]:
def num_to_letters(n):
    string = ""
    while n >= 0:
        string = chr(n % 26 + ord('A')) + string
        n = n // 26 - 1
    return string

def process_web_usage(df):
    """
    Membentuk sesi web usage 20 menit per host.
    URL diganti dengan huruf A-Z.
    """

    # Buat mapping URL → huruf
    unique_urls = df["url"].unique() # Corrected 'URL' to 'url'
    url_map = {url: num_to_letters(i) for i, url in enumerate(unique_urls)}
    df["URL_Label"] = df["url"].map(url_map) # Corrected 'URL' to 'url'

    df = df.sort_values(by=["Host", "Timestamp"])

    results = []

    for host, group in df.groupby("Host"):
        group = group.sort_values("Timestamp").reset_index(drop=True)

        sessions = []
        cur_session = []
        prev_time = None

        for _, row in group.iterrows():
            ts = row["Timestamp"]
            label = row["URL_Label"]

            if prev_time is None:
                cur_session.append((ts, label))
            else:
                delta = (ts - prev_time).total_seconds() / 60

                # aturan sesi 20 menit
                if delta > 20:
                    sessions.append(cur_session)
                    cur_session = [(ts, label)]
                else:
                    cur_session.append((ts, label))

            prev_time = ts

        if cur_session:
            sessions.append(cur_session)

        results.append({
            "Host": host,
            "Sessions": sessions
        })

    return results, url_map


# --- Hasil Langsung ---
results, url_map = process_web_usage(df)
results[:1]   # tampilkan contoh 1 host pertama

[{'Host': '***.novo.dk',
  'Sessions': [[(Timestamp('1995-07-11 12:17:09'), 'A'),
    (Timestamp('1995-07-11 12:17:11'), 'B'),
    (Timestamp('1995-07-11 12:17:31'), 'C'),
    (Timestamp('1995-07-11 12:17:33'), 'D'),
    (Timestamp('1995-07-11 12:17:34'), 'E'),
    (Timestamp('1995-07-11 12:17:38'), 'F'),
    (Timestamp('1995-07-11 12:17:48'), 'G'),
    (Timestamp('1995-07-11 12:17:51'), 'H'),
    (Timestamp('1995-07-11 12:19:13'), 'I'),
    (Timestamp('1995-07-11 12:19:17'), 'E'),
    (Timestamp('1995-07-11 12:22:03'), 'J'),
    (Timestamp('1995-07-11 12:22:08'), 'K'),
    (Timestamp('1995-07-11 12:23:01'), 'L')],
   [(Timestamp('1995-08-09 07:02:48'), 'M'),
    (Timestamp('1995-08-09 07:02:55'), 'N'),
    (Timestamp('1995-08-09 07:03:02'), 'I'),
    (Timestamp('1995-08-09 07:03:06'), 'J'),
    (Timestamp('1995-08-09 07:03:12'), 'K'),
    (Timestamp('1995-08-09 07:03:52'), 'O'),
    (Timestamp('1995-08-09 07:04:08'), 'P'),
    (Timestamp('1995-08-09 07:04:24'), 'E'),
    (Timestamp('1

##4.Membentuk Tabel Web Usage (Host – Session – Timestamp – Page)

In [ ]:
url_mapping_df = pd.DataFrame([
    {"URL": url, "Label": label}
    for url, label in url_map.items()
])

url_mapping_df

,URL,Label
0,/ksc.html,A
1,/images/ksclogo-medium.gif,B
2,/images/MOSAIC-logosmall.gif,C
3,/images/USA-logosmall.gif,D
4,/images/NASA-logosmall.gif,E
...,...,...
361,/finance/tour.gif,MX
362,/finance/brrow_1t.gif,MY
363,/shuttle/missions/sts-71/images/KSC-95EC-0423.jpg,MZ
364,/software/winvn/faq/WINVNFAQ-Contents.html,NA


##5.Ringkasan Total Sesi Per Host

In [ ]:
table_rows = []

for item in results:
    host = item["Host"]
    sessions = item["Sessions"]

    for session_idx, session in enumerate(sessions, start=1):
        for ts, label in session:
            table_rows.append({
                "Host": host,
                "Session": session_idx,
                "Timestamp": ts,
                "Page": label
            })

usage_table = pd.DataFrame(table_rows)

usage_table

,Host,Session,Timestamp,Page
0,***.novo.dk,1,1995-07-11 12:17:09,A
1,***.novo.dk,1,1995-07-11 12:17:11,B
2,***.novo.dk,1,1995-07-11 12:17:31,C
3,***.novo.dk,1,1995-07-11 12:17:33,D
4,***.novo.dk,1,1995-07-11 12:17:34,E
...,...,...,...,...
103758,128.217.62.39,13,1995-08-15 19:24:49,A
103759,128.217.62.39,13,1995-08-15 19:24:50,B
103760,128.217.62.39,13,1995-08-15 19:24:51,C
103761,128.217.62.39,13,1995-08-15 19:24:51,E


In [ ]:
session_summary = usage_table.groupby("Host")["Session"].nunique().reset_index()
session_summary.rename(columns={"Session": "Total Sessions"}, inplace=True)

session_summary

,Host,Total Sessions
0,***.novo.dk,2
1,001.msy4.communique.net,1
2,007.thegap.com,5
3,01-dynamic-c.wokingham.luna.net,6
4,01.ts01.zircon.net.au,1
...,...,...
2246,128.217.62.30,1
2247,128.217.62.33,4
2248,128.217.62.35,8
2249,128.217.62.38,19


##6.MEMBANGUN TABEL BINARY PER HALAMAN

In [ ]:
def build_binary_table(results, url_map):
    all_labels = sorted(url_map.values())  # A, B, C, ...
    rows = []

    for host_data in results:
        host = host_data["Host"]

        for session in host_data["Sessions"]:
            for ts, label in session:

                # buat baris kosong (semua 0)
                row = {l: 0 for l in all_labels}

                # halaman yang dikunjungi = 1
                row[label] = 1

                # tambahkan kolom lain
                row["Time"] = ts.strftime("%H:%M")
                row["Host"] = host

                rows.append(row)

    return pd.DataFrame(rows)


binary_table = build_binary_table(results, url_map)

print("=== Tabel Binary Halaman Dikunjungi ===")
display(binary_table.head(20))

# Simpan CSV bila diperlukan
binary_table.to_csv("binary_web_usage.csv", index=False)

=== Tabel Binary Halaman Dikunjungi ===


,A,AA,AB,AC,AD,AE,AF,AG,AH,AI,...,S,T,U,V,W,X,Y,Z,Time,Host
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:17,***.novo.dk
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:17,***.novo.dk
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:17,***.novo.dk
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:17,***.novo.dk
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:17,***.novo.dk
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:17,***.novo.dk
6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:17,***.novo.dk
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:17,***.novo.dk
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:19,***.novo.dk
9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12:19,***.novo.dk


In [ ]:
from openpyxl import Workbook

# Buat workbook baru
wb = Workbook()

# ----------------------------
# Sheet 1: URL Mapping
# ----------------------------
ws1 = wb.active
ws1.title = "URL Mapping"

# Header
ws1.append(list(url_mapping_df.columns))

# Isi data
for row in url_mapping_df.itertuples(index=False):
    ws1.append(list(row))

# ----------------------------
# Sheet 2: Web Usage Sessions
# ----------------------------
ws2 = wb.create_sheet("Web Usage Sessions")

ws2.append(list(usage_table.columns))
for row in usage_table.itertuples(index=False):
    ws2.append(list(row))

# ----------------------------
# Sheet 3: Session Summary
# ----------------------------
ws3 = wb.create_sheet("Session Summary")

ws3.append(list(session_summary.columns))
for row in session_summary.itertuples(index=False):
    ws3.append(list(row))

# ----------------------------
# Save file
# ----------------------------
output_path = "/content/web_usage_complete.xlsx"
wb.save(output_path)

print("✔ File Excel lengkap berhasil dibuat:")
print(output_path)

✔ File Excel lengkap berhasil dibuat:
/content/web_usage_complete.xlsx
